In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
import os
import re
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.patches import Rectangle, Patch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False
    print('WARNING: shap is not installed. Please install it with: pip install shap')
SEED = 42
DEVICE = 'cpu'
TASKS = ['Response', 'PFS', 'OS']
AXIS_ORDER = ['Amino Acids & TME Regulation Axis', 'Lipid & 1-Carbon Metabolism Axis', 'SCFA & BCAA Energy Metabolism Axis']
MAX_EPOCHS_FOR_EPOCH_SELECTION = 1000
PATIENCE = 80
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
DROPOUT = 0.25
LOSS_WEIGHTS = {'Response': 1.0, 'PFS': 1.0, 'OS': 1.0}
BACKGROUND_SIZE = 50
PREFER_DEEP_EXPLAINER = True
ROOT = str(project_path(get_value('science.paths.directory_008', 'directory_008')))
FIG6_FINAL_DIR = os.path.join(ROOT, get_value('science.paths.directory_163', 'directory_163'), get_value('science.paths.directory_164', 'directory_164'))
module_dir = os.path.join(FIG6_FINAL_DIR, get_value('science.paths.directory_165', 'directory_165'))
module_score_path = os.path.join(module_dir, get_value('science.tables.attribution_042', 'attribution_042.csv'))
module_meta_path = os.path.join(module_dir, get_value('science.tables.attribution_043', 'attribution_043.csv'))
label_path = os.path.join(ROOT, get_value('science.paths.directory_009', 'directory_009'), get_value('science.tables.outcomes_012', 'outcomes_012.csv'))
PY_SHAP_ROOT = os.path.join(ROOT, get_value('science.paths.directory_166', 'directory_166'))
out_dir = PY_SHAP_ROOT
fig_dir = os.path.join(out_dir, 'figures')
os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)
for name, path in [('module_score_path', module_score_path), ('module_meta_path', module_meta_path), ('label_path', label_path)]:
    pass
    






mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
OUTCOME_COLORS = {'Response': '#2D6CDF', 'PFS': '#EE854A', 'OS': '#7A4FA3'}
MODULE_TYPE_COLORS = {'Microbial KO module': '#2A9D8F', 'Metabolite module': '#D55E00', 'Radiomics module': '#4C78A8'}
SIGNED_CMAP = LinearSegmentedColormap.from_list('signed_shap', ['#3C5488', 'white', '#DC0000'])

def set_seed(seed: int=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def normalize_sample_id(x):
    return str(x).strip().lower()

def clean_label(x, max_len=42):
    x = str(x)
    x = re.sub('^KO::', '', x)
    x = re.sub('^Metab::', '', x)
    x = re.sub('^Rad::', '', x)
    x = x.replace('original_', '')
    x = x.replace('wavelet-', '')
    x = x.replace('log-sigma-', '')
    x = x.replace('_', ' ')
    x = re.sub('\\s+', ' ', x).strip()
    if len(x) > max_len:
        x = x[:max_len - 2] + '…'
    return x

def read_matrix_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig')
    df.index = df.index.map(normalize_sample_id)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def parse_binary_response(series):
    num = pd.to_numeric(series, errors='coerce')
    if num.notna().all():
        uniq = sorted(pd.unique(num.dropna()))
        if set(uniq).issubset({0, 1}):
            return num.astype(float)
    s = series.astype(str).str.strip().str.lower()
    mapping = {'1': 1.0, 'r': 1.0, 'responder': 1.0, 'response': 1.0, '0': 0.0, 'nr': 0.0, 'non-responder': 0.0, 'nonresponder': 0.0, 'non responder': 0.0}
    return s.map(mapping).astype(float)

def read_labels(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, encoding='utf-8-sig')
    possible_id_cols = ['Sample ID', 'sample_id', 'SampleID', 'sample', 'ID', 'id', 'patient_id', 'patient']
    id_col = next((c for c in possible_id_cols if c in df.columns), None)
    if id_col is not None:
        df = df.set_index(id_col)
    else:
        df = df.set_index(df.columns[0])
    df.index = df.index.map(normalize_sample_id)
    lower_cols = {c.lower(): c for c in df.columns}

    def find_col(candidates):
        for cand in candidates:
            if cand.lower() in lower_cols:
                return lower_cols[cand.lower()]
        return None
    col_map = {'Response': find_col([get_column('outcomes.response', 'Response'), 'Response', 'response', 'label', 'Responder', 'mRECIST_response', 'response_label']), 'PFS_time': find_col(['PFS_time', 'pfs_time', 'PFS', 'pfs']), 'PFS_status': find_col(['PFS_status', 'pfs_status', 'PFS_event', 'pfs_event']), 'OS_time': find_col(['OS_time', 'os_time', 'OS', 'os']), 'OS_status': find_col(['OS_status', 'os_status', 'OS_event', 'os_event'])}
    missing = [k for k, v in col_map.items() if v is None]
    if missing:
        raise ValueError(f'Missing label columns: {missing}\nAvailable columns: {list(df.columns)}')
    out = pd.DataFrame(index=df.index)
    out['Response'] = parse_binary_response(df[col_map['Response']])
    out['PFS_time'] = pd.to_numeric(df[col_map['PFS_time']], errors='coerce').astype(float)
    out['PFS_status'] = pd.to_numeric(df[col_map['PFS_status']], errors='coerce').astype(float)
    out['OS_time'] = pd.to_numeric(df[col_map['OS_time']], errors='coerce').astype(float)
    out['OS_status'] = pd.to_numeric(df[col_map['OS_status']], errors='coerce').astype(float)
    return out

def read_module_metadata(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    meta = pd.read_csv(path, encoding='utf-8-sig')
    required = ['module_id', 'display_label', 'module_type', 'row_color', 'row_order', 'axis']
    for c in required:
        if c not in meta.columns:
            raise ValueError(f'module metadata missing column: {c}')
    meta['module_id'] = meta['module_id'].astype(str)
    meta['display_label'] = meta['display_label'].astype(str)
    meta['module_type'] = meta['module_type'].astype(str)
    meta['row_color'] = meta['row_color'].astype(str)
    meta['row_order'] = pd.to_numeric(meta['row_order'], errors='coerce')
    meta['axis'] = meta['axis'].astype(str)
    return meta

def neg_cox_partial_log_likelihood(risk, time, event):
    risk = risk.view(-1)
    time = time.view(-1)
    event = event.view(-1)
    valid = torch.isfinite(time) & torch.isfinite(event)
    risk = risk[valid]
    time = time[valid]
    event = event[valid]
    if risk.numel() == 0 or event.sum() < 1:
        return torch.tensor(0.0, device=risk.device)
    order = torch.argsort(time, descending=True)
    risk_sorted = risk[order]
    event_sorted = event[order]
    log_cum_risk = torch.logcumsumexp(risk_sorted, dim=0)
    pll = risk_sorted - log_cum_risk
    loss = -torch.sum(pll * event_sorted) / (torch.sum(event_sorted) + 1e-08)
    return loss

def compute_pos_weight(y_response):
    y = np.asarray(y_response, dtype=float)
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)
    if pos <= 0:
        return 1.0
    return float(neg / pos)
set_seed(SEED)
X_raw = read_matrix_csv(module_score_path)
labels = read_labels(label_path)
module_meta = read_module_metadata(module_meta_path)
available_modules = [m for m in module_meta['module_id'].tolist() if m in X_raw.columns]
if len(available_modules) == 0:
    raise ValueError('No module IDs in metadata match columns of module score matrix.')
module_meta = module_meta[module_meta['module_id'].isin(available_modules)].copy()
module_meta = module_meta.sort_values('row_order')
X_raw = X_raw.loc[:, module_meta['module_id'].tolist()].copy()
common_samples = X_raw.index.intersection(labels.index)
X_raw = X_raw.loc[common_samples].copy()
labels = labels.loc[common_samples].copy()
valid = labels['Response'].notna() & labels['PFS_time'].notna() & labels['PFS_status'].notna() & labels['OS_time'].notna() & labels['OS_status'].notna()
X_raw = X_raw.loc[valid].copy()
labels = labels.loc[valid].copy()
print('Aligned module score matrix:', X_raw.shape)
print('Aligned labels:', labels.shape)
print(labels[['Response', 'PFS_status', 'OS_status']].sum())
X_raw.to_csv(os.path.join(out_dir, get_value('science.tables.features_017', 'features_017.csv')), encoding='utf-8-sig')
labels.to_csv(os.path.join(out_dir, get_value('science.tables.outcomes_008', 'outcomes_008.csv')), encoding='utf-8-sig')
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_imp = imputer.fit_transform(X_raw.values)
X_scaled = scaler.fit_transform(X_imp)
X_scaled_df = pd.DataFrame(X_scaled, index=X_raw.index, columns=X_raw.columns)
X_scaled_df.to_csv(os.path.join(out_dir, get_value('science.tables.features_018', 'features_018.csv')), encoding='utf-8-sig')
X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
y_response = torch.tensor(labels['Response'].values, dtype=torch.float32, device=DEVICE)
pfs_time = torch.tensor(labels['PFS_time'].values, dtype=torch.float32, device=DEVICE)
pfs_event = torch.tensor(labels['PFS_status'].values, dtype=torch.float32, device=DEVICE)
os_time = torch.tensor(labels['OS_time'].values, dtype=torch.float32, device=DEVICE)
os_event = torch.tensor(labels['OS_status'].values, dtype=torch.float32, device=DEVICE)
input_dim = X_scaled.shape[1]

class ModuleLevelMultiTaskNet(nn.Module):

    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.25):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        self.shared = nn.Sequential(*layers)
        self.response_head = nn.Linear(prev, 1)
        self.pfs_head = nn.Linear(prev, 1)
        self.os_head = nn.Linear(prev, 1)

    def forward(self, x):
        z = self.shared(x)
        response_logit = self.response_head(z).view(-1)
        pfs_risk = self.pfs_head(z).view(-1)
        os_risk = self.os_head(z).view(-1)
        return (response_logit, pfs_risk, os_risk)

def multitask_loss(model, X, y_resp, pfs_t, pfs_e, os_t, os_e, pos_weight_tensor):
    response_logit, pfs_risk, os_risk = model(X)
    loss_resp = F.binary_cross_entropy_with_logits(response_logit, y_resp, pos_weight=pos_weight_tensor)
    loss_pfs = neg_cox_partial_log_likelihood(pfs_risk, pfs_t, pfs_e)
    loss_os = neg_cox_partial_log_likelihood(os_risk, os_t, os_e)
    total = LOSS_WEIGHTS['Response'] * loss_resp + LOSS_WEIGHTS['PFS'] * loss_pfs + LOSS_WEIGHTS['OS'] * loss_os
    return (total, {'loss_total': float(total.detach().cpu()), 'loss_response': float(loss_resp.detach().cpu()), 'loss_pfs': float(loss_pfs.detach().cpu()), 'loss_os': float(loss_os.detach().cpu())})

def choose_best_epoch():
    idx = np.arange(X_scaled.shape[0])
    y_np = labels['Response'].values.astype(int)
    stratify = y_np if len(np.unique(y_np)) == 2 and min(np.bincount(y_np)) >= 3 else None
    train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=stratify)
    X_train = X_tensor_all[train_idx]
    X_val = X_tensor_all[val_idx]
    y_train = y_response[train_idx]
    y_val = y_response[val_idx]
    pfs_t_train = pfs_time[train_idx]
    pfs_e_train = pfs_event[train_idx]
    pfs_t_val = pfs_time[val_idx]
    pfs_e_val = pfs_event[val_idx]
    os_t_train = os_time[train_idx]
    os_e_train = os_event[train_idx]
    os_t_val = os_time[val_idx]
    os_e_val = os_event[val_idx]
    pos_weight = compute_pos_weight(labels['Response'].values[train_idx])
    pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)
    model = ModuleLevelMultiTaskNet(input_dim=input_dim, hidden_dims=(64, 32), dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best_epoch = 1
    best_val_loss = np.inf
    best_state = None
    patience_counter = 0
    history = []
    for epoch in range(1, MAX_EPOCHS_FOR_EPOCH_SELECTION + 1):
        model.train()
        optimizer.zero_grad()
        train_loss, train_parts = multitask_loss(model, X_train, y_train, pfs_t_train, pfs_e_train, os_t_train, os_e_train, pos_weight_tensor)
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss, val_parts = multitask_loss(model, X_val, y_val, pfs_t_val, pfs_e_val, os_t_val, os_e_val, pos_weight_tensor)
        history.append({'epoch': epoch, 'train_loss': train_parts['loss_total'], 'val_loss': val_parts['loss_total'], 'train_response': train_parts['loss_response'], 'train_pfs': train_parts['loss_pfs'], 'train_os': train_parts['loss_os'], 'val_response': val_parts['loss_response'], 'val_pfs': val_parts['loss_pfs'], 'val_os': val_parts['loss_os']})
        if val_parts['loss_total'] < best_val_loss - 1e-05:
            best_val_loss = val_parts['loss_total']
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break
    history_df = pd.DataFrame(history)
    history_df.to_csv(os.path.join(out_dir, get_value('science.tables.result_036', 'result_036.csv')), index=False, encoding='utf-8-sig')
    
    return (best_epoch, best_state, history_df)
best_epoch, best_state, history_df = choose_best_epoch()

def refit_full_model(best_epoch):
    pos_weight = compute_pos_weight(labels['Response'].values)
    pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)
    model = ModuleLevelMultiTaskNet(input_dim=input_dim, hidden_dims=(64, 32), dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    full_history = []
    for epoch in range(1, best_epoch + 1):
        model.train()
        optimizer.zero_grad()
        loss, parts = multitask_loss(model, X_tensor_all, y_response, pfs_time, pfs_event, os_time, os_event, pos_weight_tensor)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        row = {'epoch': epoch}
        row.update(parts)
        full_history.append(row)
    full_history_df = pd.DataFrame(full_history)
    full_history_df.to_csv(os.path.join(out_dir, get_value('science.tables.result_037', 'result_037.csv')), index=False, encoding='utf-8-sig')
    model.eval()
    return (model, full_history_df)
model, full_history_df = refit_full_model(best_epoch)
model_path = os.path.join(out_dir, get_value('science.artifacts.transform_006', 'transform_006.pt'))
torch.save({'model_state_dict': model.state_dict(), 'input_dim': input_dim, 'module_ids': list(X_scaled_df.columns), 'best_epoch': best_epoch, 'loss_weights': LOSS_WEIGHTS}, model_path)

model.eval()
with torch.no_grad():
    resp_logit, pfs_risk, os_risk = model(X_tensor_all)
    pred_df = pd.DataFrame({'Response_true': labels['Response'].values, 'Response_logit': resp_logit.cpu().numpy(), 'Response_pred': torch.sigmoid(resp_logit).cpu().numpy(), 'PFS_time': labels['PFS_time'].values, 'PFS_status': labels['PFS_status'].values, 'PFS_risk': pfs_risk.cpu().numpy(), 'OS_time': labels['OS_time'].values, 'OS_status': labels['OS_status'].values, 'OS_risk': os_risk.cpu().numpy()}, index=X_scaled_df.index)
pred_df.to_csv(os.path.join(out_dir, get_value('science.tables.predictions_017', 'predictions_017.csv')), encoding='utf-8-sig')
if not SHAP_AVAILABLE:
    raise ImportError('Please install shap first: pip install shap')

class TaskWrapper(nn.Module):

    def __init__(self, base_model, task):
        super().__init__()
        self.base_model = base_model
        self.task = task

    def forward(self, x):
        response_logit, pfs_risk, os_risk = self.base_model(x)
        if self.task == 'Response':
            return response_logit.view(-1, 1)
        elif self.task == 'PFS':
            return pfs_risk.view(-1, 1)
        elif self.task == 'OS':
            return os_risk.view(-1, 1)
        else:
            raise ValueError(self.task)

def extract_shap_array(shap_values):
    if isinstance(shap_values, list):
        arr = shap_values[0]
    else:
        arr = shap_values
    if torch.is_tensor(arr):
        arr = arr.detach().cpu().numpy()
    arr = np.asarray(arr)
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[:, :, 0]
    return arr

def compute_shap_for_task(task):
    wrapper = TaskWrapper(model, task).to(DEVICE)
    wrapper.eval()
    n = X_tensor_all.shape[0]
    rng = np.random.default_rng(SEED)
    bg_n = min(BACKGROUND_SIZE, n)
    bg_idx = rng.choice(np.arange(n), size=bg_n, replace=False)
    background = X_tensor_all[bg_idx]
    X_explain = X_tensor_all
    shap_values = None
    explainer_used = None
    if PREFER_DEEP_EXPLAINER:
        try:
            explainer = shap.DeepExplainer(wrapper, background)
            shap_values = explainer.shap_values(X_explain)
            explainer_used = 'DeepExplainer'
        except Exception as e:
            print(f'[{task}] DeepExplainer failed: {e}')
    if shap_values is None:
        explainer = shap.GradientExplainer(wrapper, background)
        shap_values = explainer.shap_values(X_explain)
        explainer_used = 'GradientExplainer'
    shap_arr = extract_shap_array(shap_values)
    if shap_arr.shape != X_scaled_df.shape:
        raise ValueError(f'Unexpected SHAP shape for {task}: {shap_arr.shape}, expected {X_scaled_df.shape}')
    signed_raw_df = pd.DataFrame(shap_arr, index=X_scaled_df.index, columns=X_scaled_df.columns)
    abs_df = signed_raw_df.abs()
    if task in ['PFS', 'OS']:
        signed_benefit_df = -signed_raw_df
    else:
        signed_benefit_df = signed_raw_df.copy()
    signed_raw_path = os.path.join(out_dir, f"{get_value('science.paths.directory_167', 'directory_167')}{task}.csv.gz")
    signed_benefit_path = os.path.join(out_dir, f"{get_value('science.paths.directory_168', 'directory_168')}{task}.csv.gz")
    abs_path = os.path.join(out_dir, f"{get_value('science.paths.directory_038', 'directory_038')}{task}.csv.gz")
    signed_raw_df.to_csv(signed_raw_path, compression='gzip', encoding='utf-8-sig')
    signed_benefit_df.to_csv(signed_benefit_path, compression='gzip', encoding='utf-8-sig')
    abs_df.to_csv(abs_path, compression='gzip', encoding='utf-8-sig')
    
    
    
    
    return (signed_raw_df, signed_benefit_df, abs_df, explainer_used)
shap_signed_raw = {}
shap_signed_benefit = {}
shap_abs = {}
explainer_log = []
for task in TASKS:
    signed_raw_df, signed_benefit_df, abs_df, explainer_used = compute_shap_for_task(task)
    shap_signed_raw[task] = signed_raw_df
    shap_signed_benefit[task] = signed_benefit_df
    shap_abs[task] = abs_df
    explainer_log.append({'outcome': task, 'explainer': explainer_used, 'figure_direction': 'positive = Responder-like' if task == 'Response' else 'positive = favorable / lower risk'})
pd.DataFrame(explainer_log).to_csv(os.path.join(out_dir, get_value('science.tables.attribution_017', 'attribution_017.csv')), index=False, encoding='utf-8-sig')
records = []
for task in TASKS:
    signed_raw = shap_signed_raw[task]
    signed_benefit = shap_signed_benefit[task]
    absval = shap_abs[task]
    for module_id in X_scaled_df.columns:
        meta_row = module_meta[module_meta['module_id'] == module_id]
        if meta_row.shape[0] == 0:
            display_label = module_id
            module_type = 'Unknown'
            axis = 'Unknown'
            row_color = '#BDBDBD'
            row_order = 999999.0
        else:
            display_label = str(meta_row['display_label'].iloc[0])
            module_type = str(meta_row['module_type'].iloc[0])
            axis = str(meta_row['axis'].iloc[0])
            row_color = str(meta_row['row_color'].iloc[0])
            row_order = float(meta_row['row_order'].iloc[0])
        mean_abs = float(absval[module_id].mean())
        mean_signed_raw = float(signed_raw[module_id].mean())
        mean_signed_benefit = float(signed_benefit[module_id].mean())
        records.append({'module_id': module_id, 'module_uid': module_id, 'display_label': display_label, 'module_type': module_type, 'axis': axis, 'row_color': row_color, 'row_order': row_order, 'outcome': task, 'mean_abs_shap': mean_abs, 'mean_signed_shap_raw': mean_signed_raw, 'mean_signed_shap_benefit': mean_signed_benefit, 'signed_ratio_raw': mean_signed_raw / (mean_abs + 1e-12), 'signed_ratio_benefit': mean_signed_benefit / (mean_abs + 1e-12)})
contrib = pd.DataFrame(records)
contrib['signed_ratio_raw'] = contrib['signed_ratio_raw'].clip(-1, 1)
contrib['signed_ratio_benefit'] = contrib['signed_ratio_benefit'].clip(-1, 1)
contrib['module_abs_scaled'] = contrib.groupby('outcome')['mean_abs_shap'].transform(lambda x: x / (x.max() + 1e-12))
contrib['signed_contribution_benefit'] = contrib['module_abs_scaled'] * contrib['signed_ratio_benefit']
contrib['mean_signed_shap'] = contrib['mean_signed_shap_benefit']
contrib['signed_ratio'] = contrib['signed_ratio_benefit']
contrib['signed_scaled'] = contrib['signed_contribution_benefit']
contrib = contrib.sort_values(['row_order', 'outcome'])
contrib_path = os.path.join(out_dir, get_value('science.tables.attribution_044', 'attribution_044.csv'))
contrib.to_csv(contrib_path, index=False, encoding='utf-8-sig')


def prepare_plot_data(contrib_df, include_types):
    df = contrib_df[contrib_df['module_type'].isin(include_types)].copy()
    row_df = df[['module_id', 'display_label', 'module_type', 'row_color', 'row_order']].drop_duplicates('module_id').sort_values('row_order')
    return (df, row_df)

def plot_dotmatrix(contrib_df, scope_name, include_types, show_row_names=True):
    df, row_df = prepare_plot_data(contrib_df, include_types)
    if row_df.empty:
        print(f'No rows for {scope_name}')
        return
    modules = row_df['module_id'].tolist()
    size_mat = df.pivot(index='module_id', columns='outcome', values='module_abs_scaled').reindex(index=modules, columns=TASKS).fillna(0)
    ratio_mat = df.pivot(index='module_id', columns='outcome', values='signed_ratio').reindex(index=modules, columns=TASKS).fillna(0).clip(-1, 1)
    n_rows = len(modules)
    n_cols = len(TASKS)
    fig_h = max(3.3, n_rows * 0.2 + 0.65)
    fig_w = 5.5 if show_row_names else 2.3
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    for i in range(n_rows):
        for j in range(n_cols):
            ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor='white', edgecolor='#E8E8E8', linewidth=0.5))
    for i, module_id in enumerate(modules):
        for j, task in enumerate(TASKS):
            v = float(size_mat.loc[module_id, task])
            r = float(ratio_mat.loc[module_id, task])
            if v > 0:
                ax.scatter(j, i, s=35 + 520 * np.sqrt(v), c=[r], cmap=SIGNED_CMAP, vmin=-1, vmax=1, edgecolor='#666666', linewidth=0.45, zorder=3)
    for i, (_, row) in enumerate(row_df.iterrows()):
        type_color = MODULE_TYPE_COLORS.get(row['module_type'], '#BDBDBD')
        module_color = row['row_color'] if isinstance(row['row_color'], str) else '#BDBDBD'
        ax.add_patch(Rectangle((-0.78, i - 0.45), 0.12, 0.9, facecolor=type_color, edgecolor='none', clip_on=False))
        ax.add_patch(Rectangle((-0.63, i - 0.45), 0.12, 0.9, facecolor=module_color, edgecolor='none', clip_on=False))
    ax.set_xlim(-0.9, n_cols - 0.35)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(TASKS, rotation=90, fontsize=12, fontweight='bold')
    ax.set_yticks(range(n_rows))
    if show_row_names:
        ax.set_yticklabels([clean_label(x, 42) for x in row_df['display_label']], fontsize=6.7)
    else:
        ax.set_yticklabels([''] * n_rows)
    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', length=0)
    ax.set_title('Contribution to outcome prediction', fontsize=10, fontweight='bold', pad=10)
    sm = mpl.cm.ScalarMappable(cmap=SIGNED_CMAP, norm=Normalize(vmin=-1, vmax=1))
    cbar = plt.colorbar(sm, ax=ax, fraction=0.045, pad=0.06)
    cbar.set_label('Signed\nSHAP ratio', fontsize=7)
    cbar.set_ticks([-1, 0, 1])
    cbar.set_ticklabels(['Negative', 'Neutral', 'Positive'])
    cbar.ax.tick_params(labelsize=6.5)
    size_handles = []
    size_labels = []
    for v in [0.25, 0.5, 1.0]:
        size_handles.append(ax.scatter([], [], s=35 + 520 * np.sqrt(v), c='#BDBDBD', edgecolor='#666666', linewidth=0.45))
        size_labels.append(f'{v:.2f}')
    ax.legend(size_handles, size_labels, title='Relative\nmodule |SHAP|', loc='lower left', bbox_to_anchor=(1.08, 0.02), frameon=False, fontsize=6.5, title_fontsize=7)
    plt.tight_layout(pad=0.55)
    out_png = os.path.join(fig_dir, f"{scope_name}{get_value('science.figures.plot_049', 'plot_049.png')}")
    out_pdf = os.path.join(fig_dir, f"{scope_name}{get_value('science.figures.plot_050', 'plot_050.pdf')}")
    plt.savefig(out_png, dpi=600, bbox_inches='tight', pad_inches=0.03)
    plt.savefig(out_pdf, bbox_inches='tight', pad_inches=0.03)
    plt.show()
    


In [ ]:
import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
ROOT = str(project_path(get_value('science.paths.directory_008', 'directory_008')))
FIG6_FINAL_DIR = os.path.join(ROOT, get_value('science.paths.directory_163', 'directory_163'), get_value('science.paths.directory_164', 'directory_164'))
module_dir = os.path.join(FIG6_FINAL_DIR, get_value('science.paths.directory_165', 'directory_165'))
PY_SHAP_ROOT = os.path.join(ROOT, get_value('science.paths.directory_166', 'directory_166'))
shap_dir = PY_SHAP_ROOT
contrib_path = os.path.join(shap_dir, get_value('science.tables.attribution_044', 'attribution_044.csv'))
meta_path = os.path.join(module_dir, get_value('science.tables.attribution_043', 'attribution_043.csv'))
out_dir = os.path.join(shap_dir, 'figures')
os.makedirs(out_dir, exist_ok=True)







mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
OUTCOMES = ['Response', 'PFS', 'OS']
AXIS_ORDER = ['Amino Acids & TME Regulation Axis', 'Lipid & 1-Carbon Metabolism Axis', 'SCFA & BCAA Energy Metabolism Axis']
MODULE_TYPE_ORDER = {'Microbial KO module': 0, 'Metabolite module': 1, 'Radiomics module': 2}
FIG_WIDTH = 8.2
ROW_HEIGHT = 1.18
FIG_MIN_HEIGHT = 4.25
HSPACE = 0.52
DROP_AXIS_INDEXES = []
DROP_AXIS_NAMES = []
USE_ALL_RADIOMICS_FOR_EACH_AXIS = True
USE_SIGNED_VALUE = True
SHOW_LEFT_LAYER_STRIPS = False
SHOW_MODALITY_LEGEND = False
if not os.path.exists(contrib_path):
    raise FileNotFoundError(contrib_path)
if not os.path.exists(meta_path):
    raise FileNotFoundError(meta_path)
contrib = pd.read_csv(contrib_path, encoding='utf-8-sig')
meta = pd.read_csv(meta_path, encoding='utf-8-sig')
print('\nContribution columns:')
print(list(contrib.columns))
print('\nMetadata columns:')
print(list(meta.columns))
if 'module_uid' in contrib.columns:
    contrib['module_uid_for_plot'] = contrib['module_uid'].astype(str)
else:
    contrib['module_uid_for_plot'] = contrib['module_id'].astype(str)
if 'module_uid' in meta.columns:
    meta['module_uid_for_plot'] = meta['module_uid'].astype(str)
else:
    meta['module_uid_for_plot'] = meta['module_id'].astype(str)
if 'axis' not in contrib.columns:
    contrib = contrib.merge(meta[['module_uid_for_plot', 'axis']], on='module_uid_for_plot', how='left')
required_cols = ['module_uid_for_plot', 'module_id', 'display_label', 'module_type', 'row_color', 'row_order', 'outcome']
for c in required_cols:
    if c not in contrib.columns:
        raise ValueError(f'Missing required column in contribution table: {c}')
if USE_SIGNED_VALUE:
    if 'signed_contribution_benefit' in contrib.columns:
        VALUE_COL = 'signed_contribution_benefit'
    elif 'signed_scaled' in contrib.columns:
        VALUE_COL = 'signed_scaled'
    else:
        raise ValueError('No benefit-oriented signed SHAP column found: signed_contribution_benefit or signed_scaled')
elif 'module_abs_scaled' in contrib.columns:
    VALUE_COL = 'module_abs_scaled'
elif 'mean_abs_shap' in contrib.columns:
    VALUE_COL = 'mean_abs_shap'
else:
    raise ValueError('No absolute SHAP column found: module_abs_scaled or mean_abs_shap')
print('\nUsing SHAP value column:', VALUE_COL)
for c in ['row_order', VALUE_COL]:
    contrib[c] = pd.to_numeric(contrib[c], errors='coerce')
if 'row_order' in meta.columns:
    meta['row_order'] = pd.to_numeric(meta['row_order'], errors='coerce')
RADIOMICS_MODULE_COLORS = {'Low-gray-weighted phenotype': '#5B9BD5', 'Coarse structural organization': '#7F95B8', 'Fine-scale heterogeneity': '#6F6FAE'}
for module_name, color in RADIOMICS_MODULE_COLORS.items():
    mask_contrib = (contrib['module_type'] == 'Radiomics module') & (contrib['display_label'].astype(str).str.contains(module_name, case=False, regex=False) | contrib['module_id'].astype(str).str.contains(module_name, case=False, regex=False) | contrib['module_uid_for_plot'].astype(str).str.contains(module_name, case=False, regex=False))
    contrib.loc[mask_contrib, 'row_color'] = color
    mask_meta = (meta['module_type'] == 'Radiomics module') & (meta['display_label'].astype(str).str.contains(module_name, case=False, regex=False) | meta['module_id'].astype(str).str.contains(module_name, case=False, regex=False) | meta['module_uid_for_plot'].astype(str).str.contains(module_name, case=False, regex=False))
    meta.loc[mask_meta, 'row_color'] = color

def valid_color(x, default='#BDBDBD'):
    x = str(x)
    if re.match('^#[0-9A-Fa-f]{6}$', x):
        return x
    return default

def wrap_label(x, width=42):
    x = str(x)
    x = x.replace(' Axis', '\nAxis')
    return '\n'.join(textwrap.wrap(x, width=width))

def clean_axis_name(x):
    return str(x).strip()
available_axes = set(meta.loc[meta['module_type'].isin(['Microbial KO module', 'Metabolite module']), 'axis'].dropna().astype(str))
missing_axes = [a for a in AXIS_ORDER if a not in available_axes]
if missing_axes:
    raise ValueError('Expected Figure-6 axis/axes missing from metadata: ' + ', '.join(missing_axes))
axis_order = AXIS_ORDER.copy()
if len(DROP_AXIS_NAMES) > 0:
    axis_order = [a for a in axis_order if a not in DROP_AXIS_NAMES]
if len(DROP_AXIS_INDEXES) > 0:
    axis_order = [a for i, a in enumerate(axis_order) if i not in DROP_AXIS_INDEXES]
print('\nAxes kept for plotting:')
for i, a in enumerate(axis_order):
    pass
if len(axis_order) == 0:
    raise ValueError('No biological axes remain for plotting.')
all_radiomics_modules = meta.loc[meta['module_type'] == 'Radiomics module', 'module_uid_for_plot'].dropna().astype(str).tolist()
axis_module_rows = []
for axis_name in axis_order:
    sub_meta = meta[(meta['axis'].astype(str) == axis_name) & meta['module_type'].isin(['Microbial KO module', 'Metabolite module'])].copy()
    module_uids = sub_meta['module_uid_for_plot'].astype(str).tolist()
    if USE_ALL_RADIOMICS_FOR_EACH_AXIS:
        module_uids.extend(all_radiomics_modules)
    module_uids = list(dict.fromkeys(module_uids))
    axis_module_rows.append({'axis': axis_name, 'module_uids': module_uids, 'n_modules': len(module_uids)})
axis_module_df = pd.DataFrame(axis_module_rows)
axis_map_path = os.path.join(out_dir, get_value('science.tables.result_038', 'result_038.csv'))
axis_module_df.to_csv(axis_map_path, index=False, encoding='utf-8-sig')

plot_records = []
for _, row in axis_module_df.iterrows():
    axis_name = row['axis']
    module_uids = row['module_uids']
    sub = contrib[contrib['module_uid_for_plot'].isin(module_uids) & contrib['outcome'].isin(OUTCOMES)].copy()
    if sub.empty:
        print('[Warning] Empty axis:', axis_name)
        continue
    for _, r in sub.iterrows():
        val = r[VALUE_COL]
        if not np.isfinite(val):
            continue
        plot_records.append({'axis': axis_name, 'module_uid_for_plot': r['module_uid_for_plot'], 'module_id': r['module_id'], 'display_label': r['display_label'], 'module_type': r['module_type'], 'row_color': valid_color(r['row_color']), 'row_order': r['row_order'], 'outcome': r['outcome'], 'plot_value_raw': float(val), 'abs_value_raw': abs(float(val))})
plot_df = pd.DataFrame(plot_records)
if plot_df.empty:
    raise ValueError('No plotting data generated.')
plot_df['module_type_rank'] = plot_df['module_type'].map(MODULE_TYPE_ORDER).fillna(99)
bar_total = plot_df.groupby(['axis', 'outcome'])['abs_value_raw'].sum().reset_index(name='total_abs')
global_max_total = bar_total['total_abs'].max()
if not np.isfinite(global_max_total) or global_max_total <= 0:
    global_max_total = 1.0
plot_df['plot_value'] = plot_df['plot_value_raw'] / global_max_total
plot_df['abs_value'] = plot_df['abs_value_raw'] / global_max_total
plot_table_path = os.path.join(out_dir, get_value('science.tables.metrics_017', 'metrics_017.csv'))
plot_df.to_csv(plot_table_path, index=False, encoding='utf-8-sig')

n_axes = len(axis_order)
fig = plt.figure(figsize=(FIG_WIDTH, max(FIG_MIN_HEIGHT, ROW_HEIGHT * n_axes)))
gs = fig.add_gridspec(nrows=n_axes, ncols=1, hspace=HSPACE)

def get_common_asymmetric_xlim(df, pad_fraction=0.06):
    grouped = df.groupby(['axis', 'outcome'])['plot_value']
    pos_extent = grouped.apply(lambda x: float(x[x > 0].sum()) if (x > 0).any() else 0.0).max()
    neg_extent = grouped.apply(lambda x: float(x[x < 0].sum()) if (x < 0).any() else 0.0).min()
    pos_extent = float(pos_extent) if np.isfinite(pos_extent) else 0.0
    neg_extent = float(neg_extent) if np.isfinite(neg_extent) else 0.0
    span = pos_extent - neg_extent
    if not np.isfinite(span) or span <= 0:
        span = 1.0
    left = neg_extent - pad_fraction * span if neg_extent < 0 else -0.02 * span
    right = pos_extent + pad_fraction * span if pos_extent > 0 else 0.02 * span
    return (left, right)
if USE_SIGNED_VALUE:
    xlim_left, xlim_right = get_common_asymmetric_xlim(plot_df)
else:
    xlim_max = plot_df.groupby(['axis', 'outcome'])['abs_value'].sum().max()
    if not np.isfinite(xlim_max) or xlim_max <= 0:
        xlim_max = 1.0
for i, axis_name in enumerate(axis_order):
    ax = fig.add_subplot(gs[i, 0])
    sub = plot_df[plot_df['axis'] == axis_name].copy()
    y_map = {'Response': 2, 'PFS': 1, 'OS': 0}
    ax.axvline(0, color='black', lw=0.7, zorder=1)
    for y in [0, 1, 2]:
        ax.axhline(y, color='#EFEFEF', lw=0.6, zorder=0)
    for outcome in OUTCOMES:
        y = y_map[outcome]
        sub2 = sub[sub['outcome'] == outcome].copy()
        if sub2.empty:
            continue
        sub2 = sub2.sort_values(['module_type_rank', 'row_order', 'abs_value'], ascending=[True, True, False])
        if USE_SIGNED_VALUE:
            pos_left = 0.0
            pos_sub = sub2[sub2['plot_value'] > 0].copy()
            for _, r in pos_sub.iterrows():
                width = r['plot_value']
                if not np.isfinite(width) or width <= 0:
                    continue
                ax.barh(y, width, left=pos_left, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
                pos_left += width
            neg_left = 0.0
            neg_sub = sub2[sub2['plot_value'] < 0].copy()
            for _, r in neg_sub.iterrows():
                width = abs(r['plot_value'])
                if not np.isfinite(width) or width <= 0:
                    continue
                ax.barh(y, -width, left=neg_left, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
                neg_left -= width
        else:
            right = 0.0
            for _, r in sub2.iterrows():
                width = r['abs_value']
                if not np.isfinite(width) or width <= 0:
                    continue
                ax.barh(y, width, left=right, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
                right += width
    ax.set_yticks([2, 1, 0])
    ax.set_yticklabels(['Response', 'PFS', 'OS'], fontsize=8)
    axis_title = clean_axis_name(axis_name)
    ax.set_title(axis_title, loc='left', fontsize=9, fontweight='bold', pad=3)
    if USE_SIGNED_VALUE:
        ax.set_xlim(xlim_left, xlim_right)
    else:
        ax.set_xlim(0, xlim_max * 1.05)
    ax.tick_params(axis='x', labelsize=10)
    ax.tick_params(axis='y', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel('')
fig.suptitle('Contribution to outcome prediction', fontsize=12, fontweight='bold', y=0.985)
mode_tag = 'signed' if USE_SIGNED_VALUE else 'absolute'
plt.tight_layout(rect=[0.055, 0.025, 0.99, 0.94], pad=0.55)
drop_tag = 'drop' + '_'.join(map(str, DROP_AXIS_INDEXES)) if len(DROP_AXIS_INDEXES) > 0 else 'all_axes'
out_png = os.path.join(out_dir, f"{get_value('science.paths.directory_040', 'directory_040')}{mode_tag}_{drop_tag}.png")
out_pdf = os.path.join(out_dir, f"{get_value('science.paths.directory_040', 'directory_040')}{mode_tag}_{drop_tag}.pdf")
plt.savefig(out_png, dpi=600, bbox_inches='tight', pad_inches=0.03)
plt.savefig(out_pdf, bbox_inches='tight', pad_inches=0.03)
plt.show()




In [ ]:
import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
ROOT = str(project_path(get_value('science.paths.directory_008', 'directory_008')))
FIG6_FINAL_DIR = os.path.join(ROOT, get_value('science.paths.directory_163', 'directory_163'), get_value('science.paths.directory_164', 'directory_164'))
module_dir = os.path.join(FIG6_FINAL_DIR, get_value('science.paths.directory_165', 'directory_165'))
PY_SHAP_ROOT = os.path.join(ROOT, get_value('science.paths.directory_166', 'directory_166'))
shap_dir = PY_SHAP_ROOT
meta_path = os.path.join(module_dir, get_value('science.tables.attribution_043', 'attribution_043.csv'))
mbp_path = os.path.join(ROOT, get_value('science.paths.directory_010', 'directory_010'), get_value('science.paths.directory_011', 'directory_011'), get_value('science.paths.directory_026', 'directory_026'), get_value('science.tables.result_025', 'result_025.csv'))
out_dir = os.path.join(shap_dir, 'MBP')
fig_dir = os.path.join(out_dir, 'figures')
os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)









mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
TASKS = ['Response', 'PFS', 'OS']
AXIS_ORDER = ['Amino Acids & TME Regulation Axis', 'Lipid & 1-Carbon Metabolism Axis', 'SCFA & BCAA Energy Metabolism Axis']
MBP_GROUPS = ['MBP-I', 'MBP-III']
FLIP_SURVIVAL_RISK_TO_BENEFIT = False
MODULE_TYPE_ORDER = {'Microbial KO module': 0, 'Metabolite module': 1, 'Radiomics module': 2}
USE_ALL_RADIOMICS_FOR_EACH_AXIS = True
FIG_WIDTH_SINGLE = 8.2
FIG_WIDTH_SIDE_BY_SIDE = 11.0
ROW_HEIGHT = 1.18
FIG_MIN_HEIGHT = 4.25
HSPACE = 0.52
DROP_AXIS_INDEXES = []
DROP_AXIS_NAMES = []

def normalize_sample_id(x):
    return str(x).strip().lower()

def valid_color(x, default='#BDBDBD'):
    x = str(x)
    if re.match('^#[0-9A-Fa-f]{6}$', x):
        return x
    return default

def wrap_label(x, width=48):
    x = str(x)
    x = x.replace(' Axis', '\nAxis')
    return '\n'.join(textwrap.wrap(x, width=width))

def clean_axis_name(x):
    return str(x).strip()

def safe_name(x):
    x = str(x)
    x = re.sub('[ /&:;,+()\\[\\]]+', '_', x)
    x = re.sub('_+', '_', x)
    x = re.sub('^_|_$', '', x)
    return x

def read_shap_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig')
    df.index = df.index.map(normalize_sample_id)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def get_benefit_oriented_signed(signed_df, task):
    out = signed_df.copy()
    if FLIP_SURVIVAL_RISK_TO_BENEFIT and task in ['PFS', 'OS']:
        out = -out
    return out
meta = pd.read_csv(meta_path, encoding='utf-8-sig')
required_meta_cols = ['module_id', 'display_label', 'module_type', 'row_color', 'row_order', 'axis']
for c in required_meta_cols:
    if c not in meta.columns:
        raise ValueError(f'Metadata missing column: {c}')
meta['module_id'] = meta['module_id'].astype(str)
meta['display_label'] = meta['display_label'].astype(str)
meta['module_type'] = meta['module_type'].astype(str)
meta['row_color'] = meta['row_color'].astype(str).map(valid_color)
meta['row_order'] = pd.to_numeric(meta['row_order'], errors='coerce')
meta['axis'] = meta['axis'].astype(str)
RADIOMICS_MODULE_COLORS = {'Low-gray-weighted phenotype': '#5B9BD5', 'Coarse structural organization': '#7F95B8', 'Fine-scale heterogeneity': '#6F6FAE'}
for module_name, color in RADIOMICS_MODULE_COLORS.items():
    mask_meta = (meta['module_type'] == 'Radiomics module') & (meta['display_label'].astype(str).str.contains(module_name, case=False, regex=False) | meta['module_id'].astype(str).str.contains(module_name, case=False, regex=False))
    meta.loc[mask_meta, 'row_color'] = color
mbp = pd.read_csv(mbp_path, encoding='utf-8-sig')
if 'sample' not in mbp.columns:
    raise ValueError('MBP table must contain column: sample')
if 'MBP_raw' not in mbp.columns:
    if 'MBP' in mbp.columns:
        mbp['MBP_raw'] = mbp['MBP'].astype(str).str.extract('(MBP-[I]+)', expand=False)
    else:
        raise ValueError('MBP table must contain MBP_raw or MBP.')
mbp['sample_norm'] = mbp['sample'].map(normalize_sample_id)
mbp = mbp[mbp['MBP_raw'].isin(MBP_GROUPS)].copy()
print('MBP group counts:')
print(mbp['MBP_raw'].value_counts())
shap_signed_raw = {}
shap_signed_benefit = {}
shap_abs = {}
for task in TASKS:
    signed_raw_path = os.path.join(shap_dir, f"{get_value('science.paths.directory_167', 'directory_167')}{task}.csv.gz")
    signed_benefit_path = os.path.join(shap_dir, f"{get_value('science.paths.directory_168', 'directory_168')}{task}.csv.gz")
    abs_path = os.path.join(shap_dir, f"{get_value('science.paths.directory_038', 'directory_038')}{task}.csv.gz")
    signed_raw_df = read_shap_csv(signed_raw_path)
    signed_benefit_df = read_shap_csv(signed_benefit_path)
    abs_df = read_shap_csv(abs_path)
    common_modules = [m for m in meta['module_id'].tolist() if m in signed_raw_df.columns]
    signed_raw_df = signed_raw_df[common_modules].copy()
    signed_benefit_df = signed_benefit_df[common_modules].copy()
    abs_df = abs_df[common_modules].copy()
    shap_signed_raw[task] = signed_raw_df
    shap_signed_benefit[task] = signed_benefit_df
    shap_abs[task] = abs_df
    print(task, 'signed shape:', signed_raw_df.shape)
all_shap_samples = set(shap_signed_raw['Response'].index)
for task in TASKS:
    all_shap_samples = all_shap_samples.intersection(set(shap_signed_raw[task].index))
mbp_samples = set(mbp['sample_norm'])
common_samples = sorted(list(all_shap_samples.intersection(mbp_samples)))
print('Common samples with SHAP and MBP:', len(common_samples))
if len(common_samples) == 0:
    raise ValueError('No common samples between SHAP matrices and MBP table.')
mbp_use = mbp[mbp['sample_norm'].isin(common_samples)].copy()
group_samples = {g: mbp_use.loc[mbp_use['MBP_raw'] == g, 'sample_norm'].tolist() for g in MBP_GROUPS}
for g in MBP_GROUPS:
    print(g, 'n =', len(group_samples[g]))
records = []
for group in MBP_GROUPS:
    samples = group_samples[group]
    for task in TASKS:
        signed_raw = shap_signed_raw[task].loc[samples]
        signed_benefit = shap_signed_benefit[task].loc[samples]
        absval = shap_abs[task].loc[samples]
        for module_id in signed_raw.columns:
            meta_row = meta[meta['module_id'] == module_id]
            if meta_row.shape[0] == 0:
                display_label = module_id
                module_type = 'Unknown'
                row_color = '#BDBDBD'
                row_order = 999999
                axis = 'Unknown'
            else:
                display_label = str(meta_row['display_label'].iloc[0])
                module_type = str(meta_row['module_type'].iloc[0])
                row_color = valid_color(meta_row['row_color'].iloc[0])
                row_order = float(meta_row['row_order'].iloc[0])
                axis = str(meta_row['axis'].iloc[0])
            mean_abs = float(absval[module_id].mean())
            mean_signed_raw = float(signed_raw[module_id].mean())
            mean_signed_benefit = float(signed_benefit[module_id].mean())
            records.append({'MBP_group': group, 'module_id': module_id, 'display_label': display_label, 'module_type': module_type, 'axis': axis, 'row_color': row_color, 'row_order': row_order, 'outcome': task, 'n_samples': len(samples), 'mean_abs_shap': mean_abs, 'mean_signed_shap_raw': mean_signed_raw, 'mean_signed_shap_benefit_oriented': mean_signed_benefit, 'signed_ratio_raw': mean_signed_raw / (mean_abs + 1e-12), 'signed_ratio_benefit_oriented': mean_signed_benefit / (mean_abs + 1e-12)})
contrib_group = pd.DataFrame(records)
contrib_group['signed_ratio_raw'] = contrib_group['signed_ratio_raw'].clip(-1, 1)
contrib_group['signed_ratio_benefit_oriented'] = contrib_group['signed_ratio_benefit_oriented'].clip(-1, 1)
contrib_group['module_abs_scaled_global'] = contrib_group.groupby('outcome')['mean_abs_shap'].transform(lambda x: x / (x.max() + 1e-12))
contrib_group['signed_scaled_benefit_oriented'] = contrib_group['module_abs_scaled_global'] * contrib_group['signed_ratio_benefit_oriented']
contrib_group = contrib_group.sort_values(['MBP_group', 'row_order', 'outcome'])
group_out = os.path.join(out_dir, get_value('science.tables.attribution_021', 'attribution_021.csv'))
contrib_group.to_csv(group_out, index=False, encoding='utf-8-sig')

i_df = contrib_group[contrib_group['MBP_group'] == 'MBP-I'].copy()
iii_df = contrib_group[contrib_group['MBP_group'] == 'MBP-III'].copy()
delta = i_df.merge(iii_df, on=['module_id', 'display_label', 'module_type', 'axis', 'row_color', 'row_order', 'outcome'], suffixes=('_MBP_I', '_MBP_III'))
delta['delta_mean_abs_shap_I_minus_III'] = delta['mean_abs_shap_MBP_I'] - delta['mean_abs_shap_MBP_III']
delta['delta_signed_benefit_I_minus_III'] = delta['mean_signed_shap_benefit_oriented_MBP_I'] - delta['mean_signed_shap_benefit_oriented_MBP_III']
delta['delta_signed_scaled_benefit_I_minus_III'] = delta['signed_scaled_benefit_oriented_MBP_I'] - delta['signed_scaled_benefit_oriented_MBP_III']
delta_out = os.path.join(out_dir, get_value('science.tables.attribution_022', 'attribution_022.csv'))
delta.to_csv(delta_out, index=False, encoding='utf-8-sig')

available_axes = set(meta.loc[meta['module_type'].isin(['Microbial KO module', 'Metabolite module']), 'axis'].dropna().astype(str))
missing_axes = [a for a in AXIS_ORDER if a not in available_axes]
if missing_axes:
    raise ValueError('Expected Figure-6 axis/axes missing from metadata: ' + ', '.join(missing_axes))
axis_order = AXIS_ORDER.copy()
if len(DROP_AXIS_NAMES) > 0:
    axis_order = [a for a in axis_order if a not in DROP_AXIS_NAMES]
if len(DROP_AXIS_INDEXES) > 0:
    axis_order = [a for i, a in enumerate(axis_order) if i not in DROP_AXIS_INDEXES]
print('\nBiological axes:')
for i, a in enumerate(axis_order):
    pass
all_radiomics_modules = meta.loc[meta['module_type'] == 'Radiomics module', 'module_id'].dropna().astype(str).tolist()
axis_module_rows = []
for axis_name in axis_order:
    sub_meta = meta[(meta['axis'].astype(str) == axis_name) & meta['module_type'].isin(['Microbial KO module', 'Metabolite module'])].copy()
    module_ids = sub_meta['module_id'].astype(str).tolist()
    if USE_ALL_RADIOMICS_FOR_EACH_AXIS:
        module_ids.extend(all_radiomics_modules)
    module_ids = list(dict.fromkeys(module_ids))
    axis_module_rows.append({'axis': axis_name, 'module_ids': module_ids, 'n_modules': len(module_ids)})
axis_module_df = pd.DataFrame(axis_module_rows)
axis_map_out = os.path.join(out_dir, get_value('science.tables.attribution_023', 'attribution_023.csv'))
axis_module_df.to_csv(axis_map_out, index=False, encoding='utf-8-sig')

plot_records = []
for _, row in axis_module_df.iterrows():
    axis_name = row['axis']
    module_ids = row['module_ids']
    sub = contrib_group[contrib_group['module_id'].isin(module_ids) & contrib_group['outcome'].isin(TASKS) & contrib_group['MBP_group'].isin(MBP_GROUPS)].copy()
    for _, r in sub.iterrows():
        val = r['mean_signed_shap_benefit_oriented']
        abs_val = abs(val)
        if not np.isfinite(val):
            continue
        plot_records.append({'axis': axis_name, 'MBP_group': r['MBP_group'], 'module_id': r['module_id'], 'display_label': r['display_label'], 'module_type': r['module_type'], 'row_color': valid_color(r['row_color']), 'row_order': r['row_order'], 'outcome': r['outcome'], 'plot_value_raw': float(val), 'abs_value_raw': float(abs_val)})
plot_df = pd.DataFrame(plot_records)
if plot_df.empty:
    raise ValueError('No plotting data generated.')
plot_df['module_type_rank'] = plot_df['module_type'].map(MODULE_TYPE_ORDER).fillna(99)
bar_total = plot_df.groupby(['axis', 'MBP_group', 'outcome'])['abs_value_raw'].sum().reset_index(name='total_abs')
global_max_total = bar_total['total_abs'].max()
if not np.isfinite(global_max_total) or global_max_total <= 0:
    global_max_total = 1.0
plot_df['plot_value'] = plot_df['plot_value_raw'] / global_max_total
plot_df['abs_value'] = plot_df['abs_value_raw'] / global_max_total
plot_table_out = os.path.join(out_dir, get_value('science.tables.metrics_018', 'metrics_018.csv'))
plot_df.to_csv(plot_table_out, index=False, encoding='utf-8-sig')


def draw_axis_stacked_bars(ax, sub, axis_name, title=None, show_y=True, show_axis_name=True):
    y_map = {'Response': 2, 'PFS': 1, 'OS': 0}
    ax.axvline(0, color='black', lw=0.7, zorder=1)
    for y in [0, 1, 2]:
        ax.axhline(y, color='#EFEFEF', lw=0.6, zorder=0)
    for outcome in TASKS:
        y = y_map[outcome]
        sub2 = sub[sub['outcome'] == outcome].copy()
        if sub2.empty:
            continue
        sub2 = sub2.sort_values(['module_type_rank', 'row_order', 'abs_value'], ascending=[True, True, False])
        pos_left = 0.0
        pos_sub = sub2[sub2['plot_value'] > 0].copy()
        for _, r in pos_sub.iterrows():
            width = r['plot_value']
            if not np.isfinite(width) or width <= 0:
                continue
            ax.barh(y, width, left=pos_left, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
            pos_left += width
        neg_left = 0.0
        neg_sub = sub2[sub2['plot_value'] < 0].copy()
        for _, r in neg_sub.iterrows():
            width = abs(r['plot_value'])
            if not np.isfinite(width) or width <= 0:
                continue
            ax.barh(y, -width, left=neg_left, height=0.58, color=r['row_color'], edgecolor='white', linewidth=0.5, zorder=3)
            neg_left -= width
    ax.set_yticks([2, 1, 0])
    if show_y:
        ax.set_yticklabels(['Response', 'PFS', 'OS'], fontsize=8)
    else:
        ax.set_yticklabels(['', '', ''])
    axis_title = clean_axis_name(axis_name)
    if show_axis_name:
        ax.set_title(axis_title, loc='left', fontsize=9, fontweight='bold', pad=3)
    ax.tick_params(axis='x', labelsize=10)
    ax.tick_params(axis='y', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel('')
    if title is not None:
        ax.set_title(title, loc='center', fontsize=10, fontweight='bold', pad=3)

def get_asymmetric_xlim_for_subset(df, pad_fraction=0.06):
    grouped = df.groupby(['axis', 'MBP_group', 'outcome'])['plot_value']
    pos_extent = grouped.apply(lambda x: float(x[x > 0].sum()) if (x > 0).any() else 0.0).max()
    neg_extent = grouped.apply(lambda x: float(x[x < 0].sum()) if (x < 0).any() else 0.0).min()
    pos_extent = float(pos_extent) if np.isfinite(pos_extent) else 0.0
    neg_extent = float(neg_extent) if np.isfinite(neg_extent) else 0.0
    span = pos_extent - neg_extent
    if not np.isfinite(span) or span <= 0:
        span = 1.0
    left = neg_extent - pad_fraction * span if neg_extent < 0 else -0.02 * span
    right = pos_extent + pad_fraction * span if pos_extent > 0 else 0.02 * span
    return (left, right)

def plot_single_group(group_name):
    sub_all = plot_df[plot_df['MBP_group'] == group_name].copy()
    xlim_left, xlim_right = get_asymmetric_xlim_for_subset(sub_all)
    n_axes = len(axis_order)
    fig = plt.figure(figsize=(FIG_WIDTH_SINGLE, max(FIG_MIN_HEIGHT, ROW_HEIGHT * n_axes)))
    gs = fig.add_gridspec(nrows=n_axes, ncols=1, hspace=HSPACE)
    for i, axis_name in enumerate(axis_order):
        ax = fig.add_subplot(gs[i, 0])
        sub = sub_all[sub_all['axis'] == axis_name].copy()
        draw_axis_stacked_bars(ax, sub, axis_name=axis_name, title=None, show_y=True, show_axis_name=True)
        ax.set_xlim(xlim_left, xlim_right)
    fig.suptitle(f'{group_name}: benefit-oriented module SHAP contribution', fontsize=12, fontweight='bold', y=0.985)
    plt.tight_layout(rect=[0.055, 0.025, 0.99, 0.94], pad=0.55)
    tag = safe_name(group_name)
    out_png = os.path.join(fig_dir, f"MBP_Grouped_MECFS_style_axis_summary_{tag}{get_value('science.figures.plot_051', 'plot_051.png')}")
    out_pdf = os.path.join(fig_dir, f"MBP_Grouped_MECFS_style_axis_summary_{tag}{get_value('science.figures.plot_052', 'plot_052.pdf')}")
    plt.savefig(out_png, dpi=600, bbox_inches='tight', pad_inches=0.03)
    plt.savefig(out_pdf, bbox_inches='tight', pad_inches=0.03)
    plt.show()
    
    

def plot_side_by_side():
    xlim_left, xlim_right = get_asymmetric_xlim_for_subset(plot_df)
    n_axes = len(axis_order)
    fig = plt.figure(figsize=(FIG_WIDTH_SIDE_BY_SIDE, max(FIG_MIN_HEIGHT, ROW_HEIGHT * n_axes)))
    gs = fig.add_gridspec(nrows=n_axes, ncols=2, hspace=HSPACE, wspace=0.12)
    for i, axis_name in enumerate(axis_order):
        for j, group_name in enumerate(MBP_GROUPS):
            ax = fig.add_subplot(gs[i, j])
            sub = plot_df[(plot_df['axis'] == axis_name) & (plot_df['MBP_group'] == group_name)].copy()
            draw_axis_stacked_bars(ax, sub, axis_name=axis_name, title=group_name if i == 0 else None, show_y=j == 0, show_axis_name=j == 0)
            ax.set_xlim(xlim_left, xlim_right)
    fig.suptitle('MBP-I vs MBP-III: benefit-oriented module SHAP contribution', fontsize=12, fontweight='bold', y=0.985)
    plt.tight_layout(rect=[0.045, 0.025, 0.995, 0.94], pad=0.5)
    out_png = os.path.join(fig_dir, get_value('science.figures.plot_053', 'plot_053.png'))
    out_pdf = os.path.join(fig_dir, get_value('science.figures.plot_054', 'plot_054.pdf'))
    plt.savefig(out_png, dpi=600, bbox_inches='tight', pad_inches=0.03)
    plt.savefig(out_pdf, bbox_inches='tight', pad_inches=0.03)
    plt.show()
    
    
plot_single_group('MBP-I')
plot_single_group('MBP-III')
plot_side_by_side()



